# Train 3B voice LoRA on Colab

Fresh LoRA on `Qwen/Qwen2.5-3B-Instruct` using the ETHICS **voice** CoTs.
Do **not** continue from `qwen3b-cot-sft-v2`.

**Before starting:** Runtime → Change runtime type → **T4 GPU** (A100 if you have it).

On your Mac, upload both:

- `data/training_data/synthetic_ethics_voice_cot_train.jsonl` (452)
- `data/validation_data/synthetic_ethics_voice_cot_val.jsonl` (48)

After training, the last cells run the same 100 official ETHICS cases on the **base** model and the **voice LoRA**, then download both JSONLs.

In [ ]:
import torch
assert torch.cuda.is_available(), "No GPU. Runtime → Change runtime type → T4 GPU, then rerun."
print(torch.cuda.get_device_name(0))

In [ ]:
REPO_URL = "https://github.com/vladflorinfilip/Poisoning-Chain-of-Though-Faithfulness-Through-Syntactic-Stenography.git"

!rm -rf Poisoning-Chain-of-Though-Faithfulness-Through-Syntactic-Stenography
!git clone --depth 1 "{REPO_URL}"
%cd Poisoning-Chain-of-Though-Faithfulness-Through-Syntactic-Stenography

In [ ]:
!pip install -q peft "transformers<4.50" "datasets<4" accelerate pyyaml tqdm

In [ ]:
from google.colab import files
from pathlib import Path

paths = {
    "synthetic_ethics_voice_cot_train.jsonl": Path("data/training_data/synthetic_ethics_voice_cot_train.jsonl"),
    "synthetic_ethics_voice_cot_val.jsonl": Path("data/validation_data/synthetic_ethics_voice_cot_val.jsonl"),
}
for dest in paths.values():
    dest.parent.mkdir(parents=True, exist_ok=True)

print("Upload the train JSONL, then the val JSONL (select both).")
uploaded = files.upload()
for name, data in uploaded.items():
    dest = paths.get(Path(name).name)
    if dest is None:
        raise SystemExit(f"Unexpected file {name}. Upload the train and val JSONLs.")
    dest.write_bytes(data)
    n = sum(1 for line in dest.read_text(encoding="utf-8").splitlines() if line.strip())
    print(f"wrote {dest} n={n}")

missing = [p.name for p in paths.values() if not p.exists()]
if missing:
    raise SystemExit(f"Still missing: {missing}")

In [ ]:
from pathlib import Path
import shutil

# Optional: set a Colab secret named HF_TOKEN if the Qwen download asks for auth.
# from google.colab import userdata
# import os
# os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")

train_dest = Path("data/training_data/synthetic_ethics_voice_cot_train.jsonl")
val_dest = Path("data/validation_data/synthetic_ethics_voice_cot_val.jsonl")
train_dest.parent.mkdir(parents=True, exist_ok=True)
val_dest.parent.mkdir(parents=True, exist_ok=True)

candidates = [Path("."), Path("/content"), Path("/content/Poisoning-Chain-of-Though-Faithfulness-Through-Syntactic-Stenography")]
for folder in candidates:
    src_train = folder / "synthetic_ethics_voice_cot_train.jsonl"
    src_val = folder / "synthetic_ethics_voice_cot_val.jsonl"
    if src_train.exists() and src_train.resolve() != train_dest.resolve():
        shutil.copy2(src_train, train_dest)
    if src_val.exists() and src_val.resolve() != val_dest.resolve():
        shutil.copy2(src_val, val_dest)

assert train_dest.exists(), f"Missing {train_dest}. Upload the train JSONL first."
assert val_dest.exists(), f"Missing {val_dest}. Upload the val JSONL first."
print(f"train n={sum(1 for line in train_dest.read_text().splitlines() if line.strip())}")
print(f"val   n={sum(1 for line in val_dest.read_text().splitlines() if line.strip())}")

!python training/train.py \
  --model Qwen/Qwen2.5-3B-Instruct \
  --data data/training_data/synthetic_ethics_voice_cot_train.jsonl \
  --val-data data/validation_data/synthetic_ethics_voice_cot_val.jsonl \
  --output-dir checkpoints/qwen3b-cot-sft-voice \
  --lora \
  --val-fraction 0

In [ ]:
from google.colab import files
from pathlib import Path

out = Path("checkpoints/qwen3b-cot-sft-voice")
assert (out / "adapter_model.safetensors").exists(), out
zip_path = Path("/content/qwen3b-cot-sft-voice-minimal.zip")
!zip -j {zip_path} {out}/adapter_model.safetensors {out}/adapter_config.json {out}/training_log.json
files.download(str(zip_path))

In [ ]:
from pathlib import Path
import shutil

REPO = Path("/content/Poisoning-Chain-of-Though-Faithfulness-Through-Syntactic-Stenography")
assert REPO.is_dir(), f"Missing {REPO}. Re-run the clone cell."
%cd {REPO}

dest = REPO / "evaluation" / "evaluate_ethics_morality.py"
dest.parent.mkdir(parents=True, exist_ok=True)

# files.upload() writes to the current cwd at click time, often /content.
candidates = [
    Path("/content/evaluate_ethics_morality.py"),
    Path.cwd() / "evaluate_ethics_morality.py",
    dest,
]
src = next((p for p in candidates if p.is_file()), None)
if src is None:
    from google.colab import files

    print("Upload evaluate_ethics_morality.py from your Mac.")
    uploaded = files.upload()
    if not uploaded:
        raise SystemExit("No file uploaded.")
    dest.write_bytes(uploaded[next(iter(uploaded))])
else:
    if src.resolve() != dest.resolve():
        dest.write_bytes(src.read_bytes())
    print("using", dest)

adapter = REPO / "checkpoints" / "qwen3b-cot-sft-voice"
assert (adapter / "adapter_config.json").is_file(), adapter

!python evaluation/evaluate_ethics_morality.py \
  --model Qwen/Qwen2.5-3B-Instruct \
  --device cuda \
  --limit 100 \
  --output data/evaluation_data/qwen/ETHICS/qwen3b_base.jsonl

!python evaluation/evaluate_ethics_morality.py \
  --model checkpoints/qwen3b-cot-sft-voice \
  --device cuda \
  --limit 100 \
  --output data/evaluation_data/qwen/ETHICS/qwen3b_voice.jsonl

In [ ]:
from google.colab import files

files.download("data/evaluation_data/qwen/ETHICS/qwen3b_base.jsonl")
files.download("data/evaluation_data/qwen/ETHICS/qwen3b_voice.jsonl")

## Cross-model SAE transfer screen: S1 features → voice LoRA

This is a **pre-registered exploratory transfer test**, not a new feature search.

- Reuse the frozen base-Qwen 3B SAE at **layer 27** (`d_model=2048`, dictionary `16384`).
- Treatment: the prior S1 experiment's full-dictionary combined top-6 features.
- Control: the prior S1 experiment's PEFT-enriched top-6 features (same number of directions).
- **Fixed CoT**: reuse each recorded voice-model CoT and intervene only while reading the final label.
- **Variable CoT**: intervene throughout generation from the same ETHICS prompts.

Do not select features from the voice test results: that would turn this into a within-model search rather than a cross-model transfer test.

**Interpretation caveat:** the original S1 generator also requested an active/SVO first sentence for label 0 and a passive first sentence for label 1, while the voice model uses the opposite mapping (active→1, passive→0). A positive result may therefore reflect transfer of grammatical-voice directions rather than an abstract rule-control circuit. Treat this as a transfer screen; a later voice-native ranking on held-out data is required for a mechanism claim.

In [ ]:
from pathlib import Path
import json, shutil, zipfile
import torch
from google.colab import drive, files
from transformers import AutoTokenizer

REPO = Path("/content/Poisoning-Chain-of-Though-Faithfulness-Through-Syntactic-Stenography")
assert REPO.is_dir(), "Re-run the clone cell first."
%cd {REPO}

BASE_MODEL = "Qwen/Qwen2.5-3B-Instruct"
VOICE_ADAPTER = Path("checkpoints/qwen3b-cot-sft-voice")
VOICE_GENERATIONS = Path("data/evaluation_data/qwen/ETHICS/qwen3b_voice.jsonl")
SAE_DIR = Path("sparse_autoencoders/artifacts/ethics_3b_l27")
DRIVE_SAE = Path("/content/drive/MyDrive/ethics_3b_l27/sae.pt")  # edit only if yours differs
TRANSFER_DIR = SAE_DIR / "ablations" / "voice_transfer"

# Frozen, selected only in the earlier S1 experiment.
S1_COMBINED6 = [11335, 10747, 10449, 11083, 7322, 2865]
PEFT6_CONTROL = [8413, 15754, 2403, 12753, 9711, 15074]

# Fresh runtime fallback: upload the previously downloaded minimal voice adapter.
if not (VOICE_ADAPTER / "adapter_model.safetensors").is_file():
    print("Upload qwen3b-cot-sft-voice-minimal.zip")
    uploaded = files.upload()
    archive = Path(next(iter(uploaded)))
    VOICE_ADAPTER.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(archive) as zf:
        for member in zf.namelist():
            name = Path(member).name
            if name in {"adapter_model.safetensors", "adapter_config.json", "training_log.json"}:
                (VOICE_ADAPTER / name).write_bytes(zf.read(member))
assert (VOICE_ADAPTER / "adapter_config.json").is_file(), VOICE_ADAPTER

# The minimal adapter zip has no tokenizer; it must match the original base model.
if not (VOICE_ADAPTER / "tokenizer_config.json").is_file():
    AutoTokenizer.from_pretrained(BASE_MODEL).save_pretrained(VOICE_ADAPTER)

# Fresh runtime fallback: upload the unscored qwen3b_voice.jsonl from Downloads.
if not VOICE_GENERATIONS.is_file():
    print("Upload qwen3b_voice.jsonl")
    uploaded = files.upload()
    VOICE_GENERATIONS.parent.mkdir(parents=True, exist_ok=True)
    VOICE_GENERATIONS.write_bytes(uploaded[next(iter(uploaded))])
records = [json.loads(line) for line in VOICE_GENERATIONS.read_text().splitlines() if line.strip()]
assert len(records) == 100 and len({int(r["index"]) for r in records}) == 100

# Use the exact SAE checkpoint from the S1 study; do not retrain it.
SAE_DIR.mkdir(parents=True, exist_ok=True)
sae_path = SAE_DIR / "sae.pt"
if not sae_path.is_file():
    drive.mount("/content/drive")
    assert DRIVE_SAE.is_file(), f"Missing {DRIVE_SAE}; edit DRIVE_SAE to your Drive location."
    shutil.copy2(DRIVE_SAE, sae_path)
ckpt = torch.load(sae_path, map_location="cpu", weights_only=True)
assert int(ckpt["layer"]) == 27
assert int(ckpt["dict_size"]) == 16384
assert ckpt["state_dict"]["encoder.weight"].shape[1] == 2048

TRANSFER_DIR.mkdir(parents=True, exist_ok=True)
metadata = {
    "hypothesis": "S1-derived combined directions perturb voice control more than PEFT-only directions",
    "study_type": "exploratory pre-registered cross-adapter transfer screen",
    "sae": str(sae_path), "layer": 27, "dict_size": 16384,
    "generations": str(VOICE_GENERATIONS), "n": len(records),
    "treatment_s1_full_combined6": S1_COMBINED6,
    "control_s1_peft_enriched6": PEFT6_CONTROL,
    "feature_selection_data": "S1 experiment only; no voice-test re-ranking",
    "selection_n_s1_flip_pairs": 25,
    "confound": "S1 training also tied first-sentence voice to label, with the opposite active/passive mapping",
}
(TRANSFER_DIR / "experiment.json").write_text(json.dumps(metadata, indent=2))
print(json.dumps(metadata, indent=2))

In [ ]:
import subprocess, sys

def run_ablation(name, features, mode):
    folder = "fixed_cot" if mode == "score" else "var_cot"
    out = TRANSFER_DIR / folder / f"{name}.jsonl"
    out.parent.mkdir(parents=True, exist_ok=True)
    cmd = [
        sys.executable, "-u", "sparse_autoencoders/ablate_features.py",
        "--features", *map(str, features),
        "--mode", mode,
        "--artifact-dir", str(SAE_DIR),
        "--model", str(VOICE_ADAPTER),
        "--generations", str(VOICE_GENERATIONS),
        "--output", str(out),
        "--device", "cuda",
        "--overwrite",
    ]
    print("+", " ".join(cmd), flush=True)
    subprocess.run(cmd, check=True)
    return out

# Primary readout test: the CoT is held fixed, so any label change is caused at readout.
fixed_outputs = [
    run_ablation("s1_combined6", S1_COMBINED6, "score"),
    run_ablation("peft6_control", PEFT6_CONTROL, "score"),
]
print("fixed-CoT outputs:", *fixed_outputs, sep="\n  ")

In [ ]:
# Generation test: the same intervention is active while a new CoT and label are generated.
# This is the mode where the prior S1 study found its strongest causal effect.
variable_outputs = [
    run_ablation("s1_combined6", S1_COMBINED6, "generate"),
    run_ablation("peft6_control", PEFT6_CONTROL, "generate"),
]
print("variable-CoT outputs:", *variable_outputs, sep="\n  ")

In [ ]:
# Download the four runs plus experiment.json. Score and plot locally afterward.
archive = shutil.make_archive(
    "/content/voice_transfer_3b_l27",
    "zip",
    root_dir=TRANSFER_DIR,
)
files.download(archive)
print("Downloaded", archive)